# BlindSpot: possession under missing evidence

CPU-first research pipeline: alternative histories, explicit missingness, camera evidence, fixed ablations, and a temporally separated calibration experiment. No paid GPU or hosted LLM is required.

Run cells in order. Only tactical tracking inputs are downloaded by default (~220 MB), not the 19 GB of video. Results are exploratory; public labels cover only five minutes. The calibration model is a separate experiment; the main CSV stays label-free.

In [ ]:
%pip -q install numpy==2.2.6 pandas==2.2.3 gdown==5.2.0


In [ ]:
from pathlib import Path
import hashlib, json, os, zipfile, subprocess, sys
import gdown

PROJECT = Path('/content/blindspot')
PROJECT.mkdir(exist_ok=True)
BUNDLE_ID = 'SET_AFTER_DRIVE_UPLOAD'
BUNDLE_SHA256 = 'c43d537248eb32febb4c25320487a412aee06a5f8dc0f6efac636711bb78c3e2'
def sha256(path):
    value = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            value.update(block)
    return value.hexdigest()
package = Path('/content/BlindSpot_project.zip')
if not package.exists() or sha256(package) != BUNDLE_SHA256:
    result = gdown.download(id=BUNDLE_ID, output=str(package), quiet=False)
    if result is None:
        raise RuntimeError('Bundle download failed; check sharing or Drive bandwidth limits')
assert sha256(package) == BUNDLE_SHA256, 'Project bundle checksum mismatch'
with zipfile.ZipFile(package) as archive:
    for entry in archive.infolist():
        if not (PROJECT / entry.filename).resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Unsafe bundle path')
    archive.extractall(PROJECT)
os.chdir(PROJECT)
print('Verified source bundle:', PROJECT)


In [ ]:
ASSETS = {'BAR-ATH_tactical_atd.txt': '1JEPmnLYax4dnVh1ZlInJAH5calmS1tIz', 'BAR-ATH_tactical_atd.xml': '1WBfy2sOwmJ2cX2w9XJq6y4VXo_kxKHjX', 'BAR-ATH_tactical_homography.csv': '1tuq2EGFmfvnXl8u5dU7JLoN3eNbrWwDJ', 'BAR-ATH_tactical_smart_tagging.xml': '1SvfntIRtFh6i7KAK1gVJk1NQbS_lus2K'}
EXPECTED = {'BAR-ATH_tactical_atd.txt': 'f130e350d2bfdf734b7e32f072931fccc6d015c1e837d082064875fe0a700431', 'BAR-ATH_tactical_atd.xml': '327e35e32e46a647389b88f883d4f9a2ab931e3e6148aef2fcf8f22b1e93c43a', 'BAR-ATH_tactical_homography.csv': '421c81abe16f0912dc45c2541f9bf19dd5660066397a3dcc043038a1b61c28c3', 'BAR-ATH_tactical_smart_tagging.xml': '0594d35e3df01b9c5002372d8615e2e8ef5379ac8f4e84c5ef7a53f4304c9019'}
data_dir = PROJECT / 'data/tactical'
data_dir.mkdir(parents=True, exist_ok=True)
for name, file_id in ASSETS.items():
    path = data_dir / name
    if not path.exists() or sha256(path) != EXPECTED[name]:
        result = gdown.download(id=file_id, output=str(path), quiet=False)
        if result is None:
            raise RuntimeError('Download failed: ' + name)
    assert sha256(path) == EXPECTED[name], 'Dataset checksum mismatch: ' + name
    print('Verified', name, path.stat().st_size, 'bytes')


## Mathematical and data invariants

Tests check exact decoding against exhaustive paths, posterior marginals, reproducibility, slot-order invariance, team symmetry, halftime resets, and separation of training labels.

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)


## Run the experiment

The optional calibration uses frames 11185–14184 for training, leaves a 250-frame buffer, and evaluates frames 14435–18684. Its five-minute descriptive score includes training data and must not be treated as held-out accuracy. No parameter sweep is performed. Check runtime-local storage and save the output archive after execution.

In [ ]:
subprocess.run([sys.executable, 'scripts/run_experiment.py', '--ablations', '--train-public'], check=True)


In [ ]:
OUT = PROJECT / 'data/experiments/blindspot-v1'
subprocess.run([sys.executable, 'evaluation/validate_submission.py', str(OUT / 'blindspot.csv')], check=True)
results = json.loads((OUT / 'scores.json').read_text())
import pandas as pd
display(pd.DataFrame(results).T[['macro_f1','spell_f1_mean','n_pred_spells','scope']])


## Explore evidence

The self-contained HTML report includes sampled state histories, missing-ball observations, model uncertainty, source sensitivity, query selection, and scoring provenance. Agreement is conditional on the model, not a correctness certificate. Video motion measurements require the optional video input and dependencies; the default run does not download videos.

In [ ]:
from IPython.display import HTML, display
display(HTML((OUT / 'report.html').read_text()))


## Save results before disconnecting

The archive contains the CSV, trained calibration weights, run manifest, aggregate scores, gap histories, boundary receipts, query controls, and interactive report. Keep match-derived diagnostics within the permitted sharing scope.

In [ ]:
import shutil
from google.colab import files
result_zip = shutil.make_archive('/content/BlindSpot_results', 'zip', OUT)
files.download(result_zip)
